In [1]:
# For statistical analysis
import pandas as pd, os, datetime
import numpy as np
from scipy import stats

# For plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
ehf_fpath = '/scratch/ng72/ms5578/time_series'
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
def jitter(group):
    duplicates = group.groupby(['lat', 'lon']).cumcount()
    
    # Jitter function: add a small random offset
    np.random.seed(42)  # For reproducibility
    jitter_strength = 0.05 # Adjust as needed; degrees latitude/longitude
    
    group['lat_jittered'] = group['lat'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates
    group['lon_jittered'] = group['lon'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates

    return group

In [4]:
gen_details = pd.read_csv(f"{nmap_path}/gen_info.csv")

In [5]:
# A temporary patch to fix fuel types
gen_details['fuel_source_primary'] = gen_details['fuel_source_primary'].replace({'Solar - Solar': 'Solar','Wind - Wind': 'Wind'})
gen_details = jitter(gen_details)

In [6]:
hw_tseries = pd.read_csv(f"{ehf_fpath}/gen_hw_status.csv")

In [7]:
def process_group_daily(grp, gen_fpath, hw_tseries, start_date=None, end_date=None):
    """
    Processes a single group of generator data.

    Parameters:
        grp (pd.DataFrame): A single group from gen_details (e.g., from groupby('region')).
        gen_fpath (str): Path to directory containing CSV files named by DUID (e.g., DUID.csv).
        hw_tseries (pd.DataFrame): DataFrame with columns 'time', 'DUID', and data to merge.
        start_date (str): Start date for subsetting the time series.
        end_date (str): End date for subsetting the time series.

    Returns:
        pd.DataFrame: The merged result for the group.
    """
    safe_duids = [duid.replace("/", "_").replace("\\", "_") for duid in grp['DUID']]
    gen_locs = [f"{gen_fpath}/{duid}.csv" for duid in safe_duids]
    dfs = [pd.read_csv(fp,dtype='object') for fp in gen_locs if os.path.exists(fp)]
    print(f"Loaded {len(dfs)} CSV file(s) out of {len(gen_locs)} expected.")

    if not dfs:
        return None

    # This is in case of accidental mid-file headers
    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # This is to correct the column types after removing header rows
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)
    
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    # Group by DUID and hourly time, sum TOTALMWh
    agg_func = {'TOTALMWh':'sum','TOTALCLEARED':'sum','AGCSTATUS':'max'}
    grouped = dfs.groupby(['DUID', pd.Grouper(freq='1d')]).agg(agg_func).reset_index()

    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index(['time']).sort_index()
    hw_tseries = hw_tseries.loc[sdate:edate]
    hw_tseries = hw_tseries.reset_index().set_index(['DUID','time']).sort_index()

    merged = pd.merge_asof(
        grouped.sort_values(by=['time', 'DUID']),
        hw_tseries.sort_values(by=['time', 'DUID']),
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("1d"),
        direction='nearest'
    )

    merged = merged.dropna(how='all')

    return merged

In [8]:
def select_group(gen_details, state=None, ftype=None):
    if state is not None and ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[(gen_details['region'] == state) & (gen_details['fuel_source_primary'].isin(ftype))]
        else:
            groups = gen_details.groupby(['region', 'fuel_source_primary'])
            grp = groups.get_group((state, ftype))
    elif state is not None:
        groups = gen_details.groupby('region')
        grp = groups.get_group(state)
    elif ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[gen_details['fuel_source_primary'].isin(ftype)]
        else:
            groups = gen_details.groupby('fuel_source_primary')
            grp = groups.get_group(ftype)
    else:
        grp = gen_details

    return grp


In [9]:
sdate, edate = '2009-07-01','2024-02-28'

In [10]:
def clean_df(df, gen_details): 
    df = df.merge(gen_details[['DUID', 'reg_cap_mw','technology_type_primary','fuel_source_primary','lat_jittered','lon_jittered', 'region']], on='DUID', how='left')
    df = df[~((df['TOTALMWh'] < 0))]
    mask = (df['fuel_source_primary'].isin([
            'Water', 'Natural Gas Pipeline', 'Black Coal', 'Coal Seam Methane',
            'Brown Coal', 'Diesel', 'Kerosene'
        ]) & (df['TOTALCLEARED'] <= 0))
    df.loc[mask, 'TOTALMWh'] = np.nan
    
    djf = [12, 1, 2]  # December, January, February
    df = df[df['time'].dt.month.isin(djf)]
    
    return df

In [11]:
def min_heatwave_days(df, min_days=10):
    """
    Filters the DataFrame to include only DUIDs with at least `min_days`
    of unique heatwave days (EHF_flag == 1).
    
    Parameters:
        df (pd.DataFrame): Input DataFrame with columns ['DUID', 'time', 'EHF_flag']
        min_days (int): Minimum number of unique heatwave days required

    Returns:
        pd.DataFrame: Filtered DataFrame
    """
    df = df.copy()
    df['time'] = pd.to_datetime(df['time'])
    df['date'] = df['time'].dt.date

    # Count unique heatwave days per DUID
    heatwave_days = (
        df[df['EHF_flag'] == 1]
        .groupby('DUID')['date']
        .nunique()
    )

    # Keep only DUIDs meeting the threshold
    valid_duids = heatwave_days[heatwave_days >= min_days].index
    
    return df[df['DUID'].isin(valid_duids)].copy()


In [12]:
info = select_group(gen_details,ftype=['Wind','Solar']).copy()
df = process_group_daily(info, gen_fpath, hw_tseries,sdate,edate)
df = clean_df(df, info)
df['norm_std'] = df.groupby('DUID')['TOTALMWh'].transform(lambda x: (x - x.mean()) / x.std(ddof=0))

Loaded 164 CSV file(s) out of 216 expected.


In [13]:
def hourly_anova(df, var):
    df = df.copy()
    results = []
    df['hour'] = df['time'].dt.hour
    
    for duid, group in df.groupby('DUID'):
        for hour in range(24):
            group_hour = group[group['hour'] == hour]
            
            group_0 = group_hour[group_hour['EHF_flag'] == 0][var].dropna()
            group_1 = group_hour[group_hour['EHF_flag'] == 1][var].dropna()
            
            # print(f"DUID: {duid}, hour: {hour}, size 0: {len(group_0)}, size 1: {len(group_1)}")
            
            if len(group_0) < 10 or len(group_1) < 10:
                continue
            
            t_stat, p_val = stats.f_oneway(group_0, group_1)
            
            results.append({
                'DUID': duid,
                'hour': hour,
                't_statistic': t_stat,
                'p_value': p_val,
                'mean_baseline': group_0.mean(),
                'mean_hw': group_1.mean()
            })
    
    result_df = pd.DataFrame(results)
    print("Unique hours in results:", np.unique(result_df['hour']))
    return result_df


In [14]:
def plot_hourly_anova_map(df, info, anova_results, alpha=0.05,ftypes='Wind and Solar'):
    merged = anova_results.merge(info, on='DUID', how='left').dropna(subset=['lat_jittered', 'lon_jittered'])
    merged['change'] = merged['mean_hw'] - merged['mean_baseline']
    merged['size'] = (merged['change'].abs().replace(0, 0.001) * 100) + 20

    def pvalue_significance(p):
        if p < alpha / 10:
            return 'Very strong'
        elif p < alpha:
            return 'Strong'
        elif p < 0.1:
            return 'Weak'
        else:
            return 'Not significant'

    merged['SignificanceLevel'] = merged['p_value'].apply(pvalue_significance)

    def category(row):
        sig = row['SignificanceLevel']
        if sig == 'Not significant':
            return sig
        elif row['change'] < 0:
            return f'{sig} decrease'
        else:
            return f'{sig} increase'

    merged['Category'] = merged.apply(category, axis=1)

    color_map = {
        'Very strong increase': 'darkgreen',
        'Strong increase': 'green',
        'Weak increase': 'lightgreen',
        'Very strong decrease': 'darkred',
        'Strong decrease': 'red',
        'Weak decrease': 'salmon',
        'Not significant': 'lightgray'
    }

    # Ensure 'hour' is string for animation frame to work nicely
    merged['hour'] = merged['hour'].astype(str)

    fig = px.scatter_map(
        merged,
        lat='lat_jittered',
        lon='lon_jittered',
        size='size',
        color='Category',
        color_discrete_map=color_map,
        hover_name='DUID',
        hover_data={
            'hour': True,
            'change': ':.2f',
            'mean_baseline': ':.2f',
            'mean_hw': ':.2f',
            'p_value': ':.4f',
            'lat_jittered': False,
            'lon_jittered': False,
            'size': False
        },
        zoom=5,
        map_style='carto-darkmatter',
        animation_frame='hour',
        title=f'Hourly ANOVA Results: Difference in generation on Heatwave hours for {ftypes}'
    )

    fig.update_layout(
        legend_title_text='Significance and Direction',
        margin=dict(l=10, r=10, t=50, b=10),
        map=dict(center=dict(lat=merged['lat_jittered'].mean(), lon=merged['lon_jittered'].mean()))
    )

    fig.show()
    return fig


In [15]:
df['year'] = pd.DatetimeIndex(df['time']).year

In [20]:
df.pivot_table(index=['DUID','EHF_flag'], columns='year', values='TOTALMWh', aggfunc='mean').loc['BOCORWF1']

year,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
EHF_flag,,,,,,,,,,,,,,,,
0.0,NaN,NaN,NaN,NaN,NaN,839.919306,748.676818,755.322950,986.554782,937.83114,1035.057859,861.984021,676.927976,682.629618,689.959802,703.268774
1.0,NaN,NaN,NaN,NaN,NaN,NaN,911.831111,1224.381302,1116.754833,1299.03857,965.401462,1264.403481,895.957417,NaN,819.768889,NaN


In [17]:
df.pivot_table(index=['DUID'], columns='year', values='EHF_flag', aggfunc='sum')

year,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
DUID,,,,,,,,,,,,,,,,
ADPPV1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,3.0,0.0
ARWF1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,7.0,17.0,15.0,3.0,9.0,10.0,0.0,0.0
AVLSF1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,0.0
BALDHWF1,NaN,NaN,NaN,NaN,NaN,NaN,12.0,12.0,4.0,17.0,9.0,5.0,3.0,7.0,3.0,3.0
BANGOWF1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,5.0,0.0,14.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WSTWYSF1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,14.0,0.0
WYASF1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,0.0
YARANSF1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,0.0,0.0,7.0,15.0
